In [1]:
pip install tokenizers

Note: you may need to restart the kernel to use updated packages.


In [1]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tokenizers.normalizers import NFD, StripAccents, Sequence
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing

# Special tokens for Hugging Face compatibility
special_tokens = ["<s>", "<pad>", "</s>", "<unk>", "<mask>"]

# Initialize a BPE tokenizer
tokenizer = Tokenizer(models.BPE())

# Normalization (optional, can be adjusted for Konkani)
tokenizer.normalizer = Sequence([NFD(), StripAccents()])

# Pre-tokenization (whitespace)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Trainer configuration

#'''
trainer = BpeTrainer(
    vocab_size=8000,  # Adjust as needed
    min_frequency=2,
    special_tokens=special_tokens
)

'''
trainer = BpeTrainer(
    vocab_size=16000,           # or 16000
    min_frequency=1,           # or 5
    special_tokens=special_tokens
)
'''


# Train on your single transcript file
# files = ["corpus.txt"]
files = ["all_transcripts.txt","konkani_clean.txt","corpus.txt"]
tokenizer.train(files, trainer)

# Post-processing for Hugging Face compatibility
tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> </s> $B </s>",
    special_tokens=[("<s>", 0), ("</s>", 2)]
)
tokenizer.decoder = decoders.BPEDecoder()

# Save the tokenizer
tokenizer.save("konkani-bpe-tokenizer.json")
print("Tokenizer trained and saved to konkani-bpe-tokenizer.json")


Tokenizer trained and saved to konkani-bpe-tokenizer.json


BPE ENCODING

In [2]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
from tokenizers.normalizers import NFD, StripAccents, Sequence
from tokenizers.processors import TemplateProcessing

special_tokens = ["<s>", "<pad>", "</s>", "<unk>", "<mask>"]

tokenizer = Tokenizer(models.BPE())
tokenizer.normalizer = Sequence([NFD(), StripAccents()])
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=special_tokens
)

tokenizer.train(["corpus.txt","all_transcripts.txt","konkani_clean.txt"], trainer)

tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> </s> $B </s>",
    special_tokens=[("<s>", 0), ("</s>", 2)]
)
tokenizer.decoder = decoders.BPEDecoder()

tokenizer.save("tokenizer-bpe.json")


WordPiece (used by BERT, Electra, etc.)

In [3]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tokenizer = Tokenizer(models.WordPiece(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.WordPieceTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

tokenizer.train(["corpus.txt","konkani_clean.txt","all_transcripts.txt"], trainer)
tokenizer.save("tokenizer-wordpiece.json")


Unigram Language Model (used in ALBERT, XLNet)

In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from tokenizers.normalizers import NFKC
from tokenizers.trainers import UnigramTrainer

special_tokens = ["<s>", "<pad>", "</s>", "<unk>", "<mask>"]

tokenizer = Tokenizer(models.Unigram())
tokenizer.normalizer = normalizers.Sequence([NFKC()])
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = UnigramTrainer(
    vocab_size=8000,
    special_tokens=special_tokens
)

tokenizer.train(["corpus.txt","all_transcripts.txt","konkani_clean.txt"], trainer)
tokenizer.save("tokenizer-unigram.json")


✅ 4. Google SentencePiece CLI (for production models)
📌 Hugging Face-compatible: Yes
📌 ASR-compatible: Yes
📌 Usage: Load via transformers.SentencePieceTokenizer or AutoTokenizer

In [6]:
# Install once
%pip install sentencepiece

import sentencepiece as spm

# Train SentencePiece model (Unigram or BPE)
spm.SentencePieceTrainer.train(
    input=['corpus.txt','all_transcripts.txt','konkani_clean.txt'],
    model_prefix='spm_konkani',
    vocab_size=8000,
    model_type='unigram',  # Or "bpe", "char", "word"
    character_coverage=1.0,  # Useful for Indic scripts
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    user_defined_symbols=["<mask>"]
)


Note: you may need to restart the kernel to use updated packages.


test the encoding methods

In [8]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(tokenizer_file="tokenizer-unigram.json")
encoded = hf_tokenizer.encode("हांव कालेर मोरगांव गेलो आंव भितर थोडे वेळाचेर मितरांक सोबत उबोंट आनी चाय घेतलो, तांचे संगत मजेचेर आसा.")
print("Token IDs:", encoded)
print("Tokens:", hf_tokenizer.convert_ids_to_tokens(encoded))


Token IDs: [152, 64, 857, 4396, 575, 512, 5678, 469, 4218, 405, 163, 29, 168, 14, 135, 295, 82, 9, 1254, 13, 1507, 11, 2174, 383, 21, 214, 1381, 9, 29, 255, 68, 26, 8]
Tokens: ['हांव', 'काल', 'ेर', 'मोर', 'गांव', 'गेलो', 'आंव', 'भितर', 'थोडे', 'वेळ', 'ाचेर', 'म', 'ित', 'र', 'ांक', 'सो', 'ब', 'त', 'उब', 'ो', 'ंट', 'आनी', 'चाय', 'घेतलो', ',', 'तांचे', 'संग', 'त', 'म', 'जे', 'चेर', 'आसा', '.']
